## Human In Loop 人在环中

In [9]:
from typing import TypedDict, Annotated, Literal
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.constants import START, END
from langgraph.graph import add_messages, StateGraph
from langgraph.types import Command
import os
import dotenv
from langchain.chat_models import init_chat_model

# 加载环境变量
dotenv.load_dotenv()

# 定义状态结构
class AgentState(TypedDict):
    messages: Annotated[list, add_messages]

# 初始化大模型
llm = init_chat_model(
    "deepseek-chat",
    model_provider="deepseek",
    api_key=os.getenv("DEEPSEEK_API_KEY")
)

# AI 聊天节点
def chatbot(state: AgentState):
    return {"messages": [llm.invoke(state['messages'])]}

# 人工审批节点
def human_approval(state: AgentState) -> Command[Literal["chatbot", END]]:
    question = "是否同意调用大语言模型？(y/n): "
    while True:
        response = input(question).strip().lower()
        if response in ("y", "yes"):
            return Command(goto="chatbot")
        elif response in ("n", "no"):
            print("❌ 已拒绝，流程结束。")
            return Command(goto=END)
        else:
            print("⚠️ 请输入 y 或 n。")

# 构建工作流
graph_builder = StateGraph(AgentState)
graph_builder.add_node("human_approval", human_approval)
graph_builder.add_node("chatbot", chatbot)

graph_builder.add_edge(START, "human_approval")
graph_builder.add_edge("chatbot", END)

# 编译工作流
checkpointer = InMemorySaver()
graph = graph_builder.compile(checkpointer=checkpointer)

# ======================
# 🔥 纯字符流程图（无依赖，不报错）
# ======================
print("\n" + "="*60)
print("📊 工作流字符流程图")
print("="*60)
# 核心：终端打印字符图
print(graph.get_graph().draw_ascii())
print("="*60 + "\n")

# 执行流程
if __name__ == "__main__":
    config = {"configurable": {"thread_id": "chat-1"}}
    result = graph.invoke({"messages": ["北京天气怎么样"]}, config=config)
    
    # 打印最终结果
    if "messages" in result and len(result["messages"]) > 0:
        print("\n✅ 最终回复：", result["messages"][-1].content)


📊 工作流字符流程图
          +-----------+       
          | __start__ |       
          +-----------+       
                 *            
                 *            
                 *            
        +----------------+    
        | human_approval |    
        +----------------+    
          ...         ..      
         .              ..    
       ..                 ..  
+---------+                 . 
| chatbot |               ..  
+---------+             ..    
          ***         ..      
             *      ..        
              **   .          
            +---------+       
            | __end__ |       
            +---------+       


✅ 最终回复： 我现在无法直接获取实时天气数据，建议你通过以下方式查询北京的最新天气：

1. **天气App/网站**：打开手机自带天气应用，或访问中国天气网、墨迹天气等专业平台。
2. **语音助手**：向手机AI助手（如Siri、小爱同学）提问：“北京现在天气怎么样？”
3. **搜索工具**：在百度、必应等搜索引擎直接输入“北京天气”即可看到实时信息。

如果需要了解未来几天的趋势或具体建议（如穿衣、出行），可以告诉我，我会尽力提供参考！ 🌤️


## Time Travel 时间回溯

## 综合实践

终极目标是使用LangGraph 底层 API 复现高级API create_react_agent 预构建ReACT图。

实现一个带人工审核的AI助手工作流。主要功能包括：

1. 调用模型：call_model 函数使用绑定工具的语言模型生成回复。
2. 人工审核：human_review 函数在执行工具前请求用户确认，若拒绝则终止流程。
3. 工具执行：通过 ToolNode 执行如天气查询等工具调用。
4. 状态管理：利用 StateGraph 构建节点流程，支持对话状态保存与恢复。

### 最简单的聊天机器人

In [16]:
### 最简单的聊天机器人
from typing import TypedDict, Annotated
from langgraph.constants import START, END
from langgraph.graph import add_messages, StateGraph
import os
import dotenv
from langchain.chat_models import init_chat_model
dotenv.load_dotenv(override=True)

# 定义 Agent 的状态结构，包含消息列表
class AgentState(TypedDict):
    messages: Annotated[list, add_messages]


# 初始化本地大语言模型，配置模型名称和推理模式
llm = init_chat_model(
    "deepseek-chat",
    model_provider="deepseek",
    api_key=os.getenv("DEEPSEEK_API_KEY")
)


# 聊天机器人函数，用于处理对话状态并生成回复
def chatbot(state: AgentState):
    return {"messages": [llm.invoke(state['messages'])]}


# 构建状态图结构
graph_builder = StateGraph(AgentState)

# 每个节点都与对应的处理函数进行绑定，构成工作流的基本单元
graph_builder.add_node("chatbot", chatbot)

# 添加边：从 START 到 chatbot，然后到 END
graph_builder.add_edge(START, "chatbot")
graph_builder.add_edge("chatbot",END)


# 编译图结构，并绘制可视化图表
graph = graph_builder.compile()
print("\n" + "="*50)
print("工作流字符流程图")
print("="*50)
print(graph.get_graph().draw_ascii())
print("="*50 + "\n")

response1 = graph.invoke({"messages": ["北京天气怎么样"]})

print(response1["messages"][-1].content)


工作流字符流程图
+-----------+  
| __start__ |  
+-----------+  
      *        
      *        
      *        
 +---------+   
 | chatbot |   
 +---------+   
      *        
      *        
      *        
 +---------+   
 | __end__ |   
 +---------+   

截至2025年5月，北京正处于春季向夏季过渡的时期，天气通常以晴或多云为主，气温逐渐升高，白天最高温约25-30℃，夜间约12-18℃。近期可能有轻度浮尘或短暂降雨，建议出行前查看实时预报。如果您需要更具体的当前天气信息（如温度、风力、空气质量等），请告诉我具体日期或使用天气应用查询最新数据。


### 添加提示词和工具

In [29]:
import json
import os
import dotenv
from loguru import logger
from pydantic import Field, BaseModel
from langchain_core.tools import tool

# 加载环境变量配置
dotenv.load_dotenv()


class WeatherQuery(BaseModel):
    """
    天气查询参数模型类，用于定义天气查询工具的输入参数结构。

    :param city: 城市名称，字符串类型，表示要查询天气的城市
    """
    city: str = Field(description="城市名称")


class WriteQuery(BaseModel):
    """
    写入查询模型类

    用于定义需要写入文档的内容结构，继承自BaseModel基类

    属性:
        content (str): 需要写入文档的具体内容，包含详细的描述信息
    """
    content: str = Field(description="需要写入文档的具体内容")


@tool(args_schema=WeatherQuery)
def get_weather(city):
    """
    查询指定城市的即时天气信息（模拟数据，用于学习演示）。

    :param city: 必要参数，字符串类型，表示要查询天气的城市名称。
    :return: 返回模拟的天气数据 JSON 格式字符串。
    """
    # 模拟天气数据
    mock_data = {
        "Beijing": {"city": "北京", "temp": 25, "weather": "晴", "humidity": 45},
        "Shanghai": {"city": "上海", "temp": 22, "weather": "多云", "humidity": 65},
        "Guangzhou": {"city": "广州", "temp": 28, "weather": "雷阵雨", "humidity": 80},
        "Shenzhen": {"city": "深圳", "temp": 27, "weather": "多云", "humidity": 75},
        "Hangzhou": {"city": "杭州", "temp": 23, "weather": "阴", "humidity": 60},
    }
    
    # 尝试匹配城市名（支持中英文）
    for key, data in mock_data.items():
        if key.lower() == city.lower() or data["city"] in city:
            result = json.dumps(data, ensure_ascii=False)
            logger.info(f"查询天气结果：{result}")
            return result
    
    # 默认返回北京天气
    default_data = {"city": city, "temp": 25, "weather": "晴", "humidity": 50}
    result = json.dumps(default_data, ensure_ascii=False)
    logger.info(f"查询天气结果（默认）：{result}")
    return result


@tool(args_schema=WriteQuery)
def write_file(content):
    """
    将指定内容写入本地文件

    参数:
        content (str): 要写入文件的文本内容

    返回值:
        str: 表示写入操作成功完成的提示信息
    """
    # 将内容写入res.txt文件，使用utf-8编码确保中文字符正确保存
    with open('res.txt', 'w', encoding='utf-8') as f:
        f.write(content)
        logger.info(f"已成功写入本地文件，写入内容：{content}")
        return "已成功写入本地文件。"

In [ ]:
from typing import TypedDict, Annotated
from langchain_core.messages import SystemMessage
from langgraph.constants import START, END
from langgraph.graph import add_messages, StateGraph
from langgraph.prebuilt import ToolNode
import os
import dotenv
from langchain.chat_models import init_chat_model

# 加载环境变量
dotenv.load_dotenv(override=True)

# 定义 Agent 的状态结构，包含消息列表
class AgentState(TypedDict):
    messages: Annotated[list, add_messages]


# 初始化大语言模型
llm = init_chat_model(
    "deepseek-chat",
    model_provider="deepseek",
    api_key=os.getenv("DEEPSEEK_API_KEY")
)
tools = [get_weather, write_file]
llm_with_tools = llm.bind_tools(tools)


# 聊天机器人节点，用于处理对话状态并生成回复，并告诉模型可以调用哪些工具
def chat_node(state: AgentState):
    messages = state["messages"]
    system_prompt = """你是一个智能助手，具备以下能力：
                    1. 查询天气信息
                    2. 结果写入文件
                    请根据用户的需求，选择合适的工具来完成任务。回答要准确、友好、专业。"""
    # 构建完整的消息列表（系统提示词 + 用户消息）,如果第一条消息不是系统消息，则添加系统提示词
    if not any(isinstance(msg, SystemMessage) for msg in messages):
        messages = [SystemMessage(
            content=system_prompt)] + messages
    result = llm_with_tools.invoke(messages)
    return {"messages": [result]}


# 定义工具节点（系统预置 ToolNode 会自动解析 tool_calls）
tool_node = ToolNode(tools=tools)


# 动态路由：chat_node → tool_node 或 END
def route_after_chat(state: AgentState):
    """判断是否需要进入工具节点"""
    last_message = state["messages"][-1]
    if hasattr(last_message, "tool_calls") and last_message.tool_calls:
        return "tool_node"
    return END


# 构建状态图结构
graph_builder = StateGraph(AgentState)

# 每个节点都与对应的处理函数进行绑定，构成工作流的基本单元
graph_builder.add_node("chat_node", chat_node)
graph_builder.add_node("tool_node", tool_node)

# 添加边：从 START 到 chatbot，然后到 END
graph_builder.add_edge(START, "chat_node")
# 添加条件边：根据是否有工具调用来判断是否需要进入工具节点
graph_builder.add_conditional_edges("chat_node", route_after_chat, ["tool_node", END])
# 工具节点执行完后回到 chat_node，继续多轮对话
graph_builder.add_edge("tool_node", "chat_node")

# 编译图结构，并绘制可视化图表
graph = graph_builder.compile()
print("\n" + "="*50)
print("工作流字符流程图")
print("="*50)
print(graph.get_graph().draw_ascii())
print("="*50 + "\n")

response1 = graph.invoke({"messages": ["北京天气怎么样"]})

print(response1["messages"][-1].content)


工作流字符流程图
          +-----------+             
          | __start__ |             
          +-----------+             
                 *                  
                 *                  
                 *                  
          +-----------+             
          | chat_node |             
          +-----------+             
          ...         ***           
         .               *          
       ..                 **        
+---------+           +-----------+ 
| __end__ |           | tool_node | 
+---------+           +-----------+ 



2026-05-19 10:28:33.740 | INFO     | __main__:get_weather:54 - 查询天气结果：{"city": "北京", "temp": 25, "weather": "晴", "humidity": 45}


查询到北京的天气信息如下：

| 项目 | 内容 |
|------|------|
| 🌆 **城市** | 北京 |
| 🌡️ **温度** | **25°C** |
| ☀️ **天气** | **晴** |
| 💧 **湿度** | **45%** |

北京目前天气晴朗，温度舒适宜人，湿度适中，非常适合外出活动！😊

请问您是否需要我将这些天气信息保存到文件中呢？


Failed to send compressed multipart ingest: Connection error caused failure to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. Please confirm your internet connection. ConnectionError(MaxRetryError('HTTPSConnectionPool(host=\'api.smith.langchain.com\', port=443): Max retries exceeded with url: /runs/multipart (Caused by ReadTimeoutError("HTTPSConnectionPool(host=\'api.smith.langchain.com\', port=443): Read timed out. (read timeout=3)"))'))
Content-Length: 2384
API Key: lsv2_********************************************29trace=019e3e10-3f18-78b0-8891-ed2b0014e59d,id=019e3e10-3f18-78b0-8891-ed2b0014e59d; trace=019e3e10-3f18-78b0-8891-ed2b0014e59d,id=019e3e10-3f23-7cb1-870d-bb2541ae34ad; trace=019e3e10-3f18-78b0-8891-ed2b0014e59d,id=019e3e10-3f28-7653-945f-75e501f52f4a


### 添加 HITL 环节

In [35]:
from typing import TypedDict, Annotated
import json
from langchain_core.messages import SystemMessage, ToolMessage
from langgraph.constants import START, END
from langgraph.graph import add_messages, StateGraph
from langgraph.prebuilt import ToolNode
from langgraph.checkpoint.memory import MemorySaver
import os
import dotenv
from langchain.chat_models import init_chat_model

# 加载环境变量
dotenv.load_dotenv(override=True)


# 定义 Agent 的状态结构
class AgentState(TypedDict):
    messages: Annotated[list, add_messages]


# 初始化本地大语言模型
llm = init_chat_model(
    "deepseek-chat",
    model_provider="deepseek",
    api_key=os.getenv("DEEPSEEK_API_KEY")
)
tools = [get_weather, write_file]
llm_with_tools = llm.bind_tools(tools)


# 聊天机器人节点
def chat_node(state: AgentState):
    messages = state["messages"]
    system_prompt = """你是一个智能助手，具备以下能力：
                    1. 查询天气信息
                    2. 结果写入文件
                    请根据用户的需求，选择合适的工具来完成任务。回答要准确、友好、专业。"""

    if not any(isinstance(msg, SystemMessage) for msg in messages):
        messages = [SystemMessage(content=system_prompt)] + messages

    result = llm_with_tools.invoke(messages)
    return {"messages": [result]}


# 定义工具节点
tool_node = ToolNode(tools=tools)


# 动态路由：chat_node 之后
def route_after_chat(state: AgentState):
    """判断是否需要调用工具"""
    last_message = state["messages"][-1]
    if hasattr(last_message, "tool_calls") and last_message.tool_calls:
        return "tool_node"
    return END


# 构建状态图
graph_builder = StateGraph(AgentState)

# 添加节点
graph_builder.add_node("chat_node", chat_node)
graph_builder.add_node("tool_node", tool_node)

# 添加边
graph_builder.add_edge(START, "chat_node")
graph_builder.add_conditional_edges("chat_node", route_after_chat, ["tool_node", END])
graph_builder.add_edge("tool_node", "chat_node")

# 编译图结构 - 关键：使用 interrupt_before 在工具节点前中断
memory = MemorySaver()
graph = graph_builder.compile(
    checkpointer=memory,
    interrupt_before=["tool_node"]  # 在执行工具前中断，等待人工确认
)


def run_with_approval():
    """运行带人工确认的工作流"""
    config = {"configurable": {"thread_id": "1"}}

    # 第一步：发送用户消息，执行到中断点
    print("【用户】北京天气怎么样\n")
    result = graph.invoke({"messages": ["北京天气怎么样"]}, config)

    # 检查是否中断（等待人工确认）
    snapshot = graph.get_state(config)

    if snapshot.next:  # 如果有下一个节点，说明被中断了
        print("工具调用需要人工确认")

        # 获取待执行的工具调用信息
        last_message = snapshot.values["messages"][-1]
        if hasattr(last_message, "tool_calls") and last_message.tool_calls:
            for idx, tool_call in enumerate(last_message.tool_calls, 1):
                print(f"\n[{idx}] 工具名称: {tool_call['name']}")
                print(f"    调用参数: {json.dumps(tool_call['args'], ensure_ascii=False, indent=4)}")

            approval = input("是否批准执行？(yes/no): ").strip().lower()

            if approval in ['yes', 'y']:
                print("工具调用已批准，继续执行...\n")
                # 继续执行（resume）
                result = graph.invoke(None, config)

            elif approval in ['no', 'n']:
                print("工具调用已被拒绝\n")
                # 手动添加拒绝消息，然后继续
                tool_messages = []
                for tool_call in last_message.tool_calls:
                    tool_messages.append(
                        ToolMessage(
                            content="工具调用被用户拒绝，请询问用户是否需要调整方案或提供更多信息。",
                            tool_call_id=tool_call["id"]
                        )
                    )
                # 更新状态并跳过工具节点
                graph.update_state(config, {"messages": tool_messages})
                result = graph.invoke(None, config)

    # 输出最终结果
    print("最终回复:")
    final_message = result["messages"][-1]
    print(final_message.content if hasattr(final_message, 'content') else str(final_message))


# 测试运行
if __name__ == "__main__":
    run_with_approval()

【用户】北京天气怎么样

工具调用需要人工确认

[1] 工具名称: get_weather
    调用参数: {
    "city": "北京"
}
工具调用已被拒绝

最终回复:
工具调用被用户拒绝，请询问用户是否需要调整方案或提供更多信息。


### 添加时间回溯

In [37]:
from typing import TypedDict, Annotated
from langchain_core.messages import SystemMessage, AIMessage
from langgraph.checkpoint.memory import MemorySaver
from langgraph.constants import START, END
from langgraph.graph import add_messages, StateGraph
from langgraph.prebuilt import ToolNode
import os
import dotenv
from langchain.chat_models import init_chat_model

# 加载环境变量
dotenv.load_dotenv(override=True)

# 定义 Agent 的状态结构，包含消息列表和审核状态
class AgentState(TypedDict):
    """
    描述 Agent 当前状态的数据结构。

    属性:
        messages (Annotated[list, add_messages]): 包含历史交互信息的消息列表，
                                                  使用 `add_messages` 合并新旧消息。
        user_approved (bool): 标记用户是否同意执行工具调用
    """
    messages: Annotated[list, add_messages]
    user_approved: bool


# 初始化本地大语言模型，配置基础URL、模型名称和推理模式
llm = init_chat_model(
    "deepseek-chat",
    model_provider="deepseek",
    api_key=os.getenv("DEEPSEEK_API_KEY")
)
tools = [get_weather]
model = llm.bind_tools(tools)


def call_model(state: AgentState):
    """
    调用绑定工具的大语言模型以生成响应。

    参数:
        state (AgentState): 包含当前会话中所有消息的状态对象。

    返回:
        dict: 新增模型响应后的更新状态（仅追加最新一条回复）。
    """
    system_prompt = SystemMessage("你是一个AI助手，可以依据用户提问产生回答，你还具备调用天气函数的能力")
    response = model.invoke([system_prompt] + state["messages"])
    return {"messages": [response]}


# --- 人在闭环 (HITL) 节点 ---
def human_review(state: AgentState):
    """
    在执行工具调用之前请求人工审核确认。

    如果最后一条消息包含待执行的工具调用，则提示用户进行确认。
    若用户拒绝，则终止流程；否则允许进入工具执行阶段。

    参数:
        state (AgentState): 包含当前会话状态的对象。

    返回:
        dict: 根据用户选择决定下一步操作：
              - 用户拒绝时返回系统提示消息并标记为未批准；
              - 允许继续则标记为已批准。
    """
    last_message = state["messages"][-1]

    if hasattr(last_message, "tool_calls") and last_message.tool_calls:
        call = last_message.tool_calls[0]
        tool_name = call["name"]
        tool_args = call["args"]

        print(f"[HITL] 模型计划调用工具 `{tool_name}`，参数：{tool_args}")
        confirm = input("[HITL] 是否确认执行？(y/n): ")

        if confirm.lower() != "y":
            # 用户拒绝，返回提示消息并标记为未批准
            return {
                "messages": [AIMessage(content="用户拒绝了工具调用，无法获取相关信息。")],
                "user_approved": False
            }

        # 用户同意，标记为已批准
        return {"user_approved": True}

    # 没有工具调用，直接标记为已批准
    return {"user_approved": True}


def should_review(state: AgentState):
    """
    判断是否需要进行人工审核。

    参数:
        state (AgentState): 当前状态

    返回:
        str: 下一个节点名称
    """
    last_message = state["messages"][-1]
    if hasattr(last_message, "tool_calls") and last_message.tool_calls:
        return "human_review"
    return END


def should_execute_tools(state: AgentState):
    """
    判断是否应该执行工具。

    参数:
        state (AgentState): 当前状态

    返回:
        str: 下一个节点名称
    """
    if state.get("user_approved", False):
        return "tools"
    return END


# 创建工具节点，用于执行工具调用
tool_node = ToolNode(tools)

# 构建状态图结构
graph_builder = StateGraph(AgentState)

# 每个节点都与对应的处理函数进行绑定，构成工作流的基本单元
graph_builder.add_node("agent", call_model)
graph_builder.add_node("human_review", human_review)
graph_builder.add_node("tools", tool_node)

# 添加边：从 START 到 agent
graph_builder.add_edge(START, "agent")

# 添加条件边：根据是否有工具调用来判断是否需要人工审核
graph_builder.add_conditional_edges(
    "agent",
    should_review,
    {"human_review": "human_review", END: END}
)

# 添加条件边：根据用户是否同意来决定是否执行工具或结束
graph_builder.add_conditional_edges(
    "human_review",
    should_execute_tools,
    {"tools": "tools", END: END}
)

# 工具执行完成后重新回到 agent 继续对话循环
graph_builder.add_edge("tools", "agent")

# 创建内存保存器
memory = MemorySaver()

# 编译图结构，并绘制可视化图表
graph = graph_builder.compile(checkpointer=memory)

# 配置对话线程ID
config = {"configurable": {"thread_id": "chat-1"}}

# 运行第一轮：问北京天气
print("\n" + "=" * 50)
print("第一轮对话：询问北京天气")
print("=" * 50)
response1 = graph.invoke({"messages": ["北京天气怎么样"]}, config=config)

print("\n=== 第一次结果 ===")
print(response1["messages"][-1].content)

# 打印已保存的检查点
print("\n" + "=" * 50)
print("检查点历史")
print("=" * 50)
states = list(graph.get_state_history(config))

for i, state in enumerate(states):
    print(f"\n=== 检查点 {i} (next: {state.next}) ===")
    print(f"Checkpoint ID: {state.config['configurable']['checkpoint_id']}")
    if state.values.get("messages"):
        print(f"Messages count: {len(state.values['messages'])}")

# 从第二个检查点恢复并注入新问题
print("\n" + "=" * 50)
print("第二轮对话：从检查点恢复并询问上海天气")
print("=" * 50)
new_config = graph.update_state(
    states[1].config,
    values={"messages": [{"role": "user", "content": "上海天气怎么样"}]}
)

response2 = graph.invoke(None, config=new_config)

print("\n=== 第二次结果 ===")
print(response2["messages"][-1].content)


第一轮对话：询问北京天气
[HITL] 模型计划调用工具 `get_weather`，参数：{'city': '北京'}


2026-05-19 10:35:35.161 | INFO     | __main__:get_weather:54 - 查询天气结果：{"city": "北京", "temp": 25, "weather": "晴", "humidity": 45}



=== 第一次结果 ===
北京今天的天气情况如下：

- **天气**：☀️ 晴
- **温度**：25°C
- **湿度**：45%

天气不错，是个晴朗的好天气，适合外出活动。不过湿度适中，建议多补充水分哦！有什么其他城市需要查询吗？😊

检查点历史

=== 检查点 0 (next: ()) ===
Checkpoint ID: 1f1532b6-ad74-6e50-8004-87d1839cdd5d
Messages count: 4

=== 检查点 1 (next: ('agent',)) ===
Checkpoint ID: 1f1532b6-9f65-6f38-8003-5997bfa67ff2
Messages count: 3

=== 检查点 2 (next: ('tools',)) ===
Checkpoint ID: 1f1532b6-9f4c-6197-8002-65e680872aed
Messages count: 2

=== 检查点 3 (next: ('human_review',)) ===
Checkpoint ID: 1f1532b6-420c-68b2-8001-4b571aee6ffd
Messages count: 2

=== 检查点 4 (next: ('agent',)) ===
Checkpoint ID: 1f1532b6-3648-6b93-8000-f9357d90ef3c
Messages count: 1

=== 检查点 5 (next: ('__start__',)) ===
Checkpoint ID: 1f1532b6-3581-6991-bfff-ff9953c0f9f9

第二轮对话：从检查点恢复并询问上海天气
[HITL] 模型计划调用工具 `get_weather`，参数：{'city': '上海'}


2026-05-19 10:35:57.404 | INFO     | __main__:get_weather:54 - 查询天气结果：{"city": "上海", "temp": 22, "weather": "多云", "humidity": 65}



=== 第二次结果 ===
好的！以下是查询到的两个城市的天气信息：

### 🌤 北京
- **温度**：25°C
- **天气**：☀️ 晴
- **湿度**：45%

### 🌤 上海
- **温度**：22°C
- **天气**：⛅ 多云
- **湿度**：65%

总体来看，北京天气晴朗，温度适中；上海则是多云天气，湿度稍高一些。请问还有其他城市想查询吗？


Failed to send compressed multipart ingest: Connection error caused failure to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. Please confirm your internet connection. ConnectionError(MaxRetryError('HTTPSConnectionPool(host=\'api.smith.langchain.com\', port=443): Max retries exceeded with url: /runs/multipart (Caused by ReadTimeoutError("HTTPSConnectionPool(host=\'api.smith.langchain.com\', port=443): Read timed out. (read timeout=3)"))'))
Content-Length: 3880
API Key: lsv2_********************************************29trace=019e3e16-b782-7203-acd1-210d46a17419,id=019e3e17-084b-72a1-a4df-f6ed9032f642; trace=019e3e16-b782-7203-acd1-210d46a17419,id=019e3e17-084b-72a1-a4df-f6ed9032f642; trace=019e3e16-b782-7203-acd1-210d46a17419,id=019e3e16-bc23-7e03-9f88-7cef4b6b6f15; trace=019e3e16-b782-7203-acd1-210d46a17419,id=019e3e17-0854-7441-a16a-3ed0c65c20d4; trace=019e3e16-b782-7203-acd1-210d46a17419,id=019e3e17-0858-76e3-8a41-3540a1c9b5d1; trace=019e3e16-b782-7203-acd1-210